# Poland ARE Demand Dashboard

Demand and supply-side flows from ARE **Statistical Information on the Liquid Fuels Market**,
via `scripts/update_poland.py` → `data/processed/poland/poland_are_liquid_fuels.parquet`.

## Sections
1. **Setup** — load parquet, kt → kbd / kb
2. **Headline demand** — total domestic consumption (TOTDEMO)
3. **Native products (demand)**
4. **Canonical rollup (demand)**
5. **Production & import** — REFGROUT / TOTIMPSB (kbd)
6. **Recent trends**
7. **Seasonality by year**
8. **ARE vs JODI (demand)** — TOTDEMO panels
9. **Publication timeliness** — latest ARE vs JODI month by product
10. **Commercial stocks** — CLOSTLV (Table 1.10)
11. **ARE vs JODI (stocks)** — CLOSTLV panels

## Conventions
- Native unit **kt** (tys. ton). Demand → **kbd**; stocks → **kb**.
- Demand: **Zużycie krajowe / Domestic consumption** (Tables 1.4–1.8).
- Crude excluded; five refined products only.
- All rows flagged `is_provisional=True` (monthly bulletins).

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

def _resolve_project_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "update_poland.py").exists():
            return candidate
        if (candidate / "country_oil_scraper" / "scripts" / "update_poland.py").exists():
            return candidate / "country_oil_scraper"
    raise RuntimeError(f"Could not locate project root from cwd: {here}")

PROJECT_ROOT = _resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analytics import cross_source_comparison_chart, seasonality_by_year_chart
from analytics.units import convert_series
from reference.poland import (
    ARE_STOCKS_METRIC,
    ARE_UNIT_NATIVE,
    CHART_PRODUCTS,
    DELIVERY_HEADLINE_NATIVE,
    DISPLAY_LABELS,
    JODI_COMPARE_PANEL_ORDER,
    JODI_COMPARE_SERIES,
    JODI_REF_AREA,
    JODI_STOCKS_PANEL_ORDER,
    UNITS_KIND,
    are_series_for_jodi,
    build_demand_canonical,
    build_demand_jodi_rollup,
    publication_date_from_path,
    seasonality_chart_inputs,
    timeliness_vs_jodi,
)

PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "poland" / "poland_are_liquid_fuels.parquet"
JODI_PARQUET = PROJECT_ROOT / "data" / "processed" / "jodi" / "jodi_secondary.parquet"

df = pd.read_parquet(PARQUET_PATH)
df["date"] = pd.to_datetime(df["date"])
demand = df[df["metric_type"] == "TOTDEMO"].copy()
production = df[df["metric_type"] == "REFGROUT"].copy()
imports = df[df["metric_type"] == "TOTIMPSB"].copy()
stocks = df[df["metric_type"] == ARE_STOCKS_METRIC].copy()

for frame in (demand, production, imports, stocks):
    frame["product_kind"] = frame["product_native"].map(UNITS_KIND)

demand["value_kbd"] = convert_series(
    demand["value"], ARE_UNIT_NATIVE, "kbd",
    product_kind=demand["product_kind"], date=demand["date"],
)
production["value_kbd"] = convert_series(
    production["value"], ARE_UNIT_NATIVE, "kbd",
    product_kind=production["product_kind"], date=production["date"],
)
imports["value_kbd"] = convert_series(
    imports["value"], ARE_UNIT_NATIVE, "kbd",
    product_kind=imports["product_kind"], date=imports["date"],
)
stocks["value_kb"] = convert_series(
    stocks["value"], ARE_UNIT_NATIVE, "kb",
    product_kind=stocks["product_kind"], date=stocks["date"],
)

headline = demand[demand["product_native"].isin(DELIVERY_HEADLINE_NATIVE)].copy()
demand_canonical = build_demand_canonical(demand, value_col="value_kbd")
demand_jodi_rollup = build_demand_jodi_rollup(demand_canonical, value_col="value_kbd")

bulletin_months = sorted(
    {publication_date_from_path(p) for p in df["source_file"].dropna().unique()}
)
bulletin_months = [m for m in bulletin_months if m is not None]

print(f"Loaded: {len(df):,} rows  ({df['date'].min().date()} -> {df['date'].max().date()})")
print(f"Demand rows: {len(demand):,}  |  Stock rows: {len(stocks):,}")
print(f"Bulletins cached: {df['source_file'].nunique()}  |  pub months: {len(bulletin_months)}")
print(f"Latest bulletin month: {max(bulletin_months).date() if bulletin_months else 'n/a'}")

Loaded: 396 rows  (2024-02-01 -> 2026-04-01)
Demand rows: 110  |  Stock rows: 110
Bulletins cached: 11  |  pub months: 11
Latest bulletin month: 2026-04-01


## 2. Headline — total demand (kbd)

In [2]:
headline_ts = (
    headline.groupby(["date", "is_provisional"], as_index=False)["value_kbd"]
    .sum()
    .sort_values("date")
)
fig = px.line(
    headline_ts,
    x="date",
    y="value_kbd",
    color="is_provisional",
    title="Poland total domestic consumption — ARE TOTDEMO (kbd)",
)
fig.show()

## 3. Native products (demand)

In [3]:
plot_df = demand[demand["product_native"].isin(CHART_PRODUCTS)].copy()
plot_df["label"] = plot_df["product_native"].map(DISPLAY_LABELS)
fig = px.line(
    plot_df,
    x="date",
    y="value_kbd",
    color="label",
    line_dash=plot_df["is_provisional"].map({True: "dot", False: "solid"}),
    title="Poland demand by product — ARE natives (kbd)",
)
fig.show()

## 4. JODI-aligned canonical rollup (demand)

Road diesel (**Diesel**) and heating oil (**Gasoil**) are summed into **Gas/diesel oil** (JODI GASDIES).
Section 7 canonical seasonality keeps them as separate subcategories.

In [4]:
fig_c = px.line(
    demand_jodi_rollup,
    x="date",
    y="value_kbd",
    color="panel",
    category_orders={"panel": list(demand_jodi_rollup["panel"].unique())},
    line_dash=demand_jodi_rollup["is_provisional"].map({True: "dot", False: "solid"}),
    title="Poland demand — JODI-aligned rollup (kbd; Diesel + Gasoil = GASDIES)",
)
fig_c.show()

## 5. Production & import (REFGROUT / TOTIMPSB, kbd)

In [5]:
supply_frames = []
for metric, frame, label in (
    ("REFGROUT", production, "Production"),
    ("TOTIMPSB", imports, "Import"),
):
    sl = frame[frame["product_native"].isin(CHART_PRODUCTS)].copy()
    sl["flow"] = label
    sl["label"] = sl["product_native"].map(DISPLAY_LABELS)
    supply_frames.append(sl)
supply = pd.concat(supply_frames, ignore_index=True)

fig_sup = px.line(
    supply,
    x="date",
    y="value_kbd",
    color="label",
    facet_row="flow",
    line_dash=supply["is_provisional"].map({True: "dot", False: "solid"}),
    title="Poland refinery production and import — ARE (kbd)",
)
fig_sup.update_yaxes(matches=None)
fig_sup.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_sup.show()

## 6. Recent trends (last 24 months)

In [12]:
cutoff = demand["date"].max() - pd.DateOffset(months=23)
recent = demand[demand["date"] >= cutoff].copy()
recent["label"] = recent["product_native"].map(DISPLAY_LABELS)

def _mom_yoy(g: pd.DataFrame) -> pd.Series:
    g = g.sort_values("date")
    return pd.Series({
        "last_kbd": g["value_kbd"].iloc[-1],
        "mom_pct": (g["value_kbd"].iloc[-1] / g["value_kbd"].iloc[-2] - 1) * 100
        if len(g) >= 2 else np.nan,
        "yoy_pct": (g["value_kbd"].iloc[-1] / g["value_kbd"].iloc[-13] - 1) * 100
        if len(g) >= 13 else np.nan,
    })

snap = (
    recent.groupby("product_native", group_keys=False)
    .apply(_mom_yoy, include_groups=False)
    .reset_index()
)
snap["label"] = snap["product_native"].map(DISPLAY_LABELS)
display(snap.sort_values("last_kbd", ascending=False))

,product_native,last_kbd,mom_pct,yoy_pct,label
0,Diesel oils,406.162958,-4.383379,8.325261,Diesel
4,Motor gasoline,160.216613,16.911325,29.730730,Gasoline
3,LPG,74.266177,-8.977775,-8.921025,LPG
2,Heating oil,8.336276,-31.792899,-54.102089,Heating oil
1,Fuel oil,3.918344,11.201795,-33.945010,Fuel oil


## 7. Seasonality by year

In [7]:
DEFAULT_SEASONALITY_VIEW = "native"
view_picker = widgets.Dropdown(
    options=[("Native products", "native"), ("Canonical", "canonical")],
    value=DEFAULT_SEASONALITY_VIEW,
    description="View:",
)

def plot_seasonality(view: str = DEFAULT_SEASONALITY_VIEW) -> None:
    season_df, product_col, products, labels, suffix = seasonality_chart_inputs(
        demand, demand_canonical, view=view, value_col="value_kbd"
    )
    if season_df.empty:
        print("[skip] No rows for seasonality.")
        return
    fig = seasonality_by_year_chart(
        season_df,
        products,
        product_col=product_col,
        value_col="value_kbd",
        product_labels=labels,
        default_visible_prior_years=5,
        units_label="kbd",
        title=f"Poland demand — seasonality ({suffix})",
    )
    fig.show()

widgets.interact(plot_seasonality, view=view_picker)

interactive(children=(Dropdown(description='View:', options=(('Native products', 'native'), ('Canonical', 'can…

<function __main__.plot_seasonality(view: str = 'native') -> None>

## 8. ARE vs JODI (PL, TOTDEMO, kbd)

In [8]:
if not JODI_PARQUET.exists():
    print(f"[skip] JODI parquet not found: {JODI_PARQUET}")
    print("       Run: python scripts/update_jodi.py")
else:
    jodi = pd.read_parquet(JODI_PARQUET)
    jodi["date"] = pd.to_datetime(jodi["date"])
    jodi_lookup = {spec.jodi_energy_product: spec.panel for spec in JODI_COMPARE_SERIES.values()}
    jodi_codes = set(jodi_lookup)

    are_panels = []
    for key in JODI_COMPARE_SERIES:
        sl = are_series_for_jodi(demand, key, value_col="value_kbd")
        if sl.empty:
            continue
        spec = JODI_COMPARE_SERIES[key]
        are_panels.append(sl.assign(panel=spec.panel))
    are_panel = pd.concat(are_panels, ignore_index=True) if are_panels else pd.DataFrame()

    jodi_pl = jodi[
        (jodi["ref_area"] == JODI_REF_AREA)
        & (jodi["flow_breakdown"] == "TOTDEMO")
        & (jodi["unit_measure"] == "KBD")
        & (jodi["energy_product"].isin(jodi_codes))
    ].copy()
    jodi_pl["panel"] = jodi_pl["energy_product"].map(jodi_lookup)
    jodi_pl["value_kbd"] = jodi_pl["obs_value"]

    panels = [p for p in JODI_COMPARE_PANEL_ORDER if p in set(are_panel.get("panel", []))]
    if not panels:
        print("[skip] No overlapping JODI panels.")
    else:
        fig = cross_source_comparison_chart(
            df_a=are_panel,
            df_b=jodi_pl,
            products=panels,
            product_col_a="panel",
            product_col_b="panel",
            value_col_a="value_kbd",
            value_col_b="value_kbd",
            label_a="ARE",
            label_b="JODI",
            title="Poland TOTDEMO — ARE vs JODI (kbd)",
            units_label="kbd",
        )
        fig.show()

## 9. Publication timeliness — ARE vs JODI latest month

Positive **lead_months** means ARE carries data further than JODI secondary for that product.
Diesel and Gasoil both map to JODI **GASDIES** (not split in JODI PL).

In [9]:
if not JODI_PARQUET.exists():
    print("[skip] JODI parquet missing")
else:
    jodi = pd.read_parquet(JODI_PARQUET)
    jodi["date"] = pd.to_datetime(jodi["date"])
    jodi_pl_demo = jodi[
        (jodi["ref_area"] == JODI_REF_AREA)
        & (jodi["flow_breakdown"] == "TOTDEMO")
        & (jodi["unit_measure"] == "KBD")
    ]
    tl = timeliness_vs_jodi(demand, jodi_pl_demo)
    display(tl)

    overlap_end = min(demand["date"].max(), jodi_pl_demo["date"].max())
    are_gas = demand[
        (demand["product_native"] == "Motor gasoline") & (demand["date"] == overlap_end)
    ]
    jodi_gas = jodi_pl_demo[
        (jodi_pl_demo["energy_product"] == "GASOLINE") & (jodi_pl_demo["date"] == overlap_end)
    ]
    if not are_gas.empty and not jodi_gas.empty:
        diff = float(are_gas["value_kbd"].iloc[0]) - float(jodi_gas["obs_value"].iloc[0])
        print(
            f"Gasoline {overlap_end:%Y-%m}: ARE {are_gas['value_kbd'].iloc[0]:.1f} kbd "
            f"vs JODI {jodi_gas['obs_value'].iloc[0]:.1f} kbd  (delta {diff:+.1f} kbd)"
        )

,panel,jodi_energy_product,are_latest,jodi_latest,lead_months
0,Gasoline,GASOLINE,2026-04-01,2026-04-01,0
1,Diesel,GASDIES,2026-04-01,2026-04-01,0
2,Gasoil,GASDIES,2026-04-01,2026-04-01,0
3,LPG,LPG,2026-04-01,2026-04-01,0
4,Fuel oil,RESFUEL,2026-04-01,2026-04-01,0


Gasoline 2026-04: ARE 160.2 kbd vs JODI 161.1 kbd  (delta -0.9 kbd)


## 10. Commercial stocks (CLOSTLV, kb)

In [10]:
if stocks.empty:
    print("[skip] No CLOSTLV rows — run: python scripts/update_poland.py --bootstrap")
else:
    plot_stk = stocks[stocks["product_native"].isin(CHART_PRODUCTS)].copy()
    plot_stk["label"] = plot_stk["product_native"].map(DISPLAY_LABELS)
    fig_stk = px.line(
        plot_stk,
        x="date",
        y="value_kb",
        color="label",
        line_dash=plot_stk["is_provisional"].map({True: "dot", False: "solid"}),
        title="Poland commercial stocks — ARE Table 1.10 (kb)",
    )
    fig_stk.show()

    latest = stocks["date"].max()
    latest_stk = (
        stocks[stocks["date"] == latest]
        .assign(label=lambda d: d["product_native"].map(DISPLAY_LABELS))
        .sort_values("value_kb", ascending=False)
    )
    display(latest_stk[["label", "value", "value_kb", "is_provisional"]])

,label,value,value_kb,is_provisional
378,Diesel,1074.3502,8014.652492,True
382,Gasoline,199.2421,1693.557850,True
381,LPG,129.6311,1503.720760,True
380,Heating oil,70.2015,523.703190,True
379,Fuel oil,35.7582,238.149612,True


## 11. ARE vs JODI (CLOSTLV, kb)

In [11]:
if stocks.empty:
    print("[skip] No CLOSTLV rows in parquet")
elif not JODI_PARQUET.exists():
    print("[skip] JODI parquet missing")
else:
    are_stk_panels = []
    for key in JODI_COMPARE_SERIES:
        sl = are_series_for_jodi(stocks, key, value_col="value_kb")
        if sl.empty:
            continue
        spec = JODI_COMPARE_SERIES[key]
        are_stk_panels.append(sl.assign(panel=spec.panel))
    are_stk_panel = pd.concat(are_stk_panels, ignore_index=True) if are_stk_panels else pd.DataFrame()

    jodi_stk = jodi[
        (jodi["ref_area"] == JODI_REF_AREA)
        & (jodi["flow_breakdown"] == "CLOSTLV")
        & (jodi["unit_measure"] == "KBBL")
        & (jodi["energy_product"].isin(jodi_codes))
    ].copy()
    jodi_stk["panel"] = jodi_stk["energy_product"].map(jodi_lookup)
    jodi_stk["value_kb"] = jodi_stk["obs_value"]

    stk_panels = [p for p in JODI_STOCKS_PANEL_ORDER if p in set(are_stk_panel.get("panel", []))]
    if not stk_panels:
        print("[skip] No overlapping JODI stock panels.")
    else:
        fig_stk_cmp = cross_source_comparison_chart(
            df_a=are_stk_panel,
            df_b=jodi_stk,
            products=stk_panels,
            product_col_a="panel",
            product_col_b="panel",
            value_col_a="value_kb",
            value_col_b="value_kb",
            label_a="ARE",
            label_b="JODI",
            title="Poland CLOSTLV — ARE vs JODI (kb)",
            units_label="kb",
        )
        fig_stk_cmp.show()